- This script is will be used to collect the Petrinex CSV files, combine
- The files into a DataFrame and filter for the company and download the volumetrics for that operating company

In [13]:
# THIS SECTION OF THE CODE IS USED TO COMBINE ALL THE DATA DOWNLOADED
#FROM PETRINEX INTO ONE BIG DATAFRAME 
import importlib
import pandas as pd
import glob 
import os
import numpy as np
from emissions_production_calc import *

In [16]:
#Variables 
peterinex_folder_path = '/Users/moadmin/Desktop/Programming_Projects/Petrinex_Analysis/Petrinex Source Data/Volumetrics/Vol_2022-01-AB.CSV'
operator_name = 'I3 ENERGY CANADA LTD.'

In [21]:
# Load thr dataframe from the CSV
test_df = pd.read_csv(peterinex_folder_path)

# Convert month ProductionMonth to datatime
test_df['ProductionMonth'] = pd.to_datetime(test_df['ProductionMonth'])

# Add the Year and Month columns
test_df['Year'] = test_df['ProductionMonth'].dt.year
test_df['Month'] = test_df['ProductionMonth'].dt.strftime('%b')

# Filter for the correct Operator
 = test_df[test_df['OperatorName'] == operator_name]

# Filter on the ActivityID to activities that resulting in emissions released
test_df = test_df[(test_df['ActivityID'] == 'FUEL') | (test_df['ActivityID'] == 'FLARE')]


In [22]:
#Add Columns for emission factor for flare and stationary fuel combustion
# Add CO2 Emission Factor
test_df['CO2_Emission_Factor'] = np.where(test_df['ActivityID'] == 'FUEL', 0.00233,
                                          np.where(test_df['ActivityID'] == 'FLARE', 2280, 0.0))
#Add CH4 Emission Factor 
test_df['CH4_Emission_Factor'] = np.where(test_df['ActivityID'] == 'FUEL', 6.4E-06,
                                          np.where(test_df['ActivityID'] == 'FLARE', 10.83, 0.0))
#Add N2O Emissions Factors
test_df['N2O_Emission_Factor'] = np.where(test_df['ActivityID'] == 'FUEL', 6.0E-08,
                                          np.where(test_df['ActivityID'] == 'FLARE', 0.033, 0.0))
# Add CO2 Emission Factor units
test_df['CO2_Emission_Factor_Unit'] = np.where(test_df['ActivityID'] == 'FUEL', 'tonnes/m3',
                                          np.where(test_df['ActivityID'] == 'FLARE', 'g/m3',''))
#Add CH4 Emission Factor units
test_df['CH4_Emission_Factor_Unit'] = np.where(test_df['ActivityID'] == 'FUEL', 'tonnes/m3',
                                          np.where(test_df['ActivityID'] == 'FLARE', 'g/m3',''))
#Add N2O Emissions Factors units
test_df['N2O_Emission_Factor_Unit'] = np.where(test_df['ActivityID'] == 'FUEL', 'tonnes/m3',
                                          np.where(test_df['ActivityID'] == 'FLARE', 'g/m3',''))

#Add Columns for emission factor for flare and stationary fuel combustion
test_df[['CO2_Emissions_tonne'
        ,'CH4_Emissions_tonne'
        ,'N2O_Emissions_tonne'
        ,'Total_Emissions_TCO2e']] = ''


#filtered dataframe
filtered_test_df = test_df[['Year'
    ,'Month'
    ,'OperatorBAID'
    ,'OperatorName'
    ,'ReportingFacilityID'
    ,'ReportingFacilityName'
    ,'ReportingFacilityLocation'
    ,'ActivityID'
    ,'ProductID'
    ,'Volume'
    ,'CO2_Emission_Factor'
    ,'CO2_Emission_Factor_Unit'
    ,'CO2_Emissions_tonne'
    ,'CH4_Emission_Factor'
    ,'CH4_Emission_Factor_Unit'
    ,'CH4_Emissions_tonne'
    ,'N2O_Emission_Factor'
    ,'N2O_Emission_Factor_Unit'
    ,'N2O_Emissions_tonne'
    ,'Total_Emissions_TCO2e']]




In [26]:
#Create a new dataframe for emissions 
emissions_df_instance = ConOilnGasEmissionCalc(filtered_test_df)

emissions_final = emissions_df_instance.calculate_emissions(
                                                            fuel_type='ActivityID',
                                                            volume_col='Volume', 
                                                            co2_ef='CO2_Emission_Factor',
                                                            ch4_ef='CH4_Emission_Factor',
                                                            n2o_ef='N2O_Emission_Factor',
                                                            co2_result='CO2_Emissions_tonne',
                                                            ch4_result='CH4_Emissions_tonne',
                                                            n2o_result='N2O_Emissions_tonne',
                                                            total_emission='Total_Emissions_TCO2e'
                                                            )

emissions_final.head()

/Users/moadmin/Desktop/Programming_Projects/Petrinex_Analysis/Scripts/emissions_production_calc.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.emissions_df[co2_result] = numpy.where(
/Users/moadmin/Desktop/Programming_Projects/Petrinex_Analysis/Scripts/emissions_production_calc.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.emissions_df[ch4_result] = numpy.where(
/Users/moadmin/Desktop/Programming_Projects/Petrinex_Analysis/Scripts/emissions_production_calc.py:38: SettingWithCopyWarnin

,Year,Month,OperatorBAID,OperatorName,ReportingFacilityID,ReportingFacilityName,ReportingFacilityLocation,ActivityID,ProductID,Volume,CO2_Emission_Factor,CO2_Emission_Factor_Unit,CO2_Emissions_tonne,CH4_Emission_Factor,CH4_Emission_Factor_Unit,CH4_Emissions_tonne,N2O_Emission_Factor,N2O_Emission_Factor_Unit,N2O_Emissions_tonne,Total_Emissions_TCO2e
547086,2022,Jan,A8HW,I3 ENERGY CANADA LTD.,ABBT0042948,GULF WESTEROSE 06-28,06-28-044-01W5,FUEL,GAS,0.4,0.00233,tonnes/m3,0.932,0.000006,tonnes/m3,0.00256,6.000000e-08,tonnes/m3,0.000024,1.01004
547092,2022,Jan,A8HW,I3 ENERGY CANADA LTD.,ABBT0043304,PHILLIPS WESTROSE SOUTH 11-21,11-21-044-01W5,FUEL,GAS,0.5,0.00233,tonnes/m3,1.165,0.000006,tonnes/m3,0.00320,6.000000e-08,tonnes/m3,0.000030,1.26255
547103,2022,Jan,A8HW,I3 ENERGY CANADA LTD.,ABBT0043439,POCO PEMBINA 06-28,06-28-049-11W5,FUEL,GAS,0.7,0.00233,tonnes/m3,1.631,0.000006,tonnes/m3,0.00448,6.000000e-08,tonnes/m3,0.000042,1.76757
547104,2022,Jan,A8HW,I3 ENERGY CANADA LTD.,ABBT0043439,POCO PEMBINA 06-28,06-28-049-11W5,FUEL,GAS,0.9,0.00233,tonnes/m3,2.097,0.000006,tonnes/m3,0.00576,6.000000e-08,tonnes/m3,0.000054,2.27259
547105,2022,Jan,A8HW,I3 ENERGY CANADA LTD.,ABBT0043439,POCO PEMBINA 06-28,06-28-049-11W5,FUEL,GAS,0.4,0.00233,tonnes/m3,0.932,0.000006,tonnes/m3,0.00256,6.000000e-08,tonnes/m3,0.000024,1.01004


In [27]:
test_df.head()

,ProductionMonth,OperatorBAID,OperatorName,ReportingFacilityID,ReportingFacilityProvinceState,ReportingFacilityType,ReportingFacilityIdentifier,ReportingFacilityName,ReportingFacilitySubType,ReportingFacilitySubTypeDesc,...,CO2_Emission_Factor,CH4_Emission_Factor,N2O_Emission_Factor,CO2_Emission_Factor_Unit,CH4_Emission_Factor_Unit,N2O_Emission_Factor_Unit,CO2_Emissions_tonne,CH4_Emissions_tonne,N2O_Emissions_tonne,Total_Emissions_TCO2e
547086,2022-01-01,A8HW,I3 ENERGY CANADA LTD.,ABBT0042948,AB,BT,42948,GULF WESTEROSE 06-28,351,GAS SINGLE WELL BATTERY,...,0.00233,0.000006,6.000000e-08,tonnes/m3,tonnes/m3,tonnes/m3,,,,
547092,2022-01-01,A8HW,I3 ENERGY CANADA LTD.,ABBT0043304,AB,BT,43304,PHILLIPS WESTROSE SOUTH 11-21,351,GAS SINGLE WELL BATTERY,...,0.00233,0.000006,6.000000e-08,tonnes/m3,tonnes/m3,tonnes/m3,,,,
547103,2022-01-01,A8HW,I3 ENERGY CANADA LTD.,ABBT0043439,AB,BT,43439,POCO PEMBINA 06-28,361,GAS MULTIWELL GROUP BATTERY,...,0.00233,0.000006,6.000000e-08,tonnes/m3,tonnes/m3,tonnes/m3,,,,
547104,2022-01-01,A8HW,I3 ENERGY CANADA LTD.,ABBT0043439,AB,BT,43439,POCO PEMBINA 06-28,361,GAS MULTIWELL GROUP BATTERY,...,0.00233,0.000006,6.000000e-08,tonnes/m3,tonnes/m3,tonnes/m3,,,,
547105,2022-01-01,A8HW,I3 ENERGY CANADA LTD.,ABBT0043439,AB,BT,43439,POCO PEMBINA 06-28,361,GAS MULTIWELL GROUP BATTERY,...,0.00233,0.000006,6.000000e-08,tonnes/m3,tonnes/m3,tonnes/m3,,,,


In [3]:
# Specify the folder path consisting the CSV files
folder_path = '/Users/moadmin/Desktop/Programming Projects/Petrinex Analysis/Petrinex Source Data'
#list out all the files in the folder
files = os.listdir(folder_path)
# files # Display the file names

In [ ]:
# Initiallize an empty DataFrame to store the combined data
combined_data = pd.DataFrame()

# Look through the files in the folder 
for file_name in files: 
    file_path = os.path.join(folder_path, file_name)

    #Check if the file is a CSV file
    if file_name.endswith('.CSV') and os.path.isfile(file_path):
        try:
            #read the CSV file into a DataFrame
            df = pd.read_csv(file_path)

            # Append the dataframe to the combined_data DataFrame
            combined_data = combined_data.append(df, ignore_index = True)
            print(f"Read data from '{file_name} and appended to the Combined Data DataFrame")   # This line of code is not needed
        except Exception as e:
            print(f"Error reading '{file_name}: {e}")

# Print the combined DataFrame
# print("Combined Data:")
# print(combined_data)


In [ ]:
# Convert month ProductionMonth to datatime
combined_data['ProductionMonth'] = pd.to_datetime(combined_data['ProductionMonth'])

# Add the Year and Month columns
combined_data['Year'] = combined_data['ProductionMonth'].dt.year
combined_data['Month'] = combined_data['ProductionMonth'].dt.strftime('%b')

In [36]:
# Filter on the ActivityID to activities that resulting in emissions released
emissions_df = combined_data[(combined_data['ActivityID'] == 'FUEL') | (combined_data['ActivityID'] == 'FLARE')]

#Add Columns for emission factor for flare and stationary fuel combustion
emissions_df[['CO2_Emission_Factor'
        ,'CO2_Emission_Factor_Unit'
        ,'CO2_Emissions_tonne'
        ,'CH4_Emission_Factor'
        ,'CH4_Emission_Factor_Unit'
        ,'CH4_Emissions_tonne'
        ,'N2O_Emission_Factor'
        ,'N2O_Emission_Factor_Unit'
        ,'N2O_Emissions_tonne'
        ,'Total_Emissions_TCO2e']] = ''


#filtered dataframe
filtered_emissions_df = emissions_df[['Year'
    ,'Month'
    ,'OperatorBAID'
    ,'OperatorName'
    ,'ReportingFacilityID'
    ,'ReportingFacilityName'
    ,'ReportingFacilityLocation'
    ,'ActivityID'
    ,'ProductID'
    ,'Volume'
    ,'CO2_Emission_Factor'
    ,'CO2_Emission_Factor_Unit'
    ,'CO2_Emissions_tonne'
    ,'CH4_Emission_Factor'
    ,'CH4_Emission_Factor_Unit'
    ,'CH4_Emissions_tonne'
    ,'N2O_Emission_Factor'
    ,'N2O_Emission_Factor_Unit'
    ,'N2O_Emissions_tonne'
    ,'Total_Emissions_TCO2e']]

filtered_emissions_df.head()

NameError: name 'combined_data' is not defined

This section filters the company 
while True: 
    if operator_name in combined_data['OperatorName'].values:
        print("The Operator You're Filtering For Is: " + operator_name)
        break
    else:
        print("We did not find you operator name in our list. Please enter the Operator Name as it Appears in Petrinex, Your Entry is:" + operator_name)

In [24]:
filtered_company = combined_data[combined_data['OperatorName']== operator_name]
filtered_company.head(5)

,ProductionMonth,OperatorBAID,OperatorName,ReportingFacilityID,ReportingFacilityProvinceState,ReportingFacilityType,ReportingFacilityIdentifier,ReportingFacilityName,ReportingFacilitySubType,ReportingFacilitySubTypeDesc,...,FromToIDProvinceState,FromToIDType,FromToIDIdentifier,Volume,Energy,Hours,CCICode,ProrationProduct,ProrationFactor,Heat
393836,2022-05,A6GH,HALO EXPLORATION LTD.,ABBT0151348,AB,BT,151348,HALO MCKINLEY 03-22-065-22W5,321,CRUDE OIL MULTIWELL GROUP BATTERY,...,NaN,NaN,NaN,0.0,NaN,NaN,,NaN,NaN,NaN
393837,2022-05,A6GH,HALO EXPLORATION LTD.,ABBT0151348,AB,BT,151348,HALO MCKINLEY 03-22-065-22W5,321,CRUDE OIL MULTIWELL GROUP BATTERY,...,AB,BT,0146653,2411.4,NaN,NaN,,NaN,NaN,NaN
393838,2022-05,A6GH,HALO EXPLORATION LTD.,ABBT0151348,AB,BT,151348,HALO MCKINLEY 03-22-065-22W5,321,CRUDE OIL MULTIWELL GROUP BATTERY,...,AB,BT,0151348,392.6,NaN,NaN,,NaN,NaN,NaN
393839,2022-05,A6GH,HALO EXPLORATION LTD.,ABBT0151348,AB,BT,151348,HALO MCKINLEY 03-22-065-22W5,321,CRUDE OIL MULTIWELL GROUP BATTERY,...,AB,BT,0151348,199.5,NaN,NaN,,NaN,NaN,NaN
393840,2022-05,A6GH,HALO EXPLORATION LTD.,ABBT0151348,AB,BT,151348,HALO MCKINLEY 03-22-065-22W5,321,CRUDE OIL MULTIWELL GROUP BATTERY,...,AB,WI,100022006522W500,529.4,NaN,NaN,,NaN,NaN,NaN


#THIS SECTION FILTERS MULTIPLE CRITERIA

filtered_company_list = ['JAPAN CANADA OIL SANDS LIMITED','GREENFIRE HANGINGSTONE OPERATING CORPORATION','GREENFIRE ACQUISITION CORPORATION','GREENFIRE RESOURCES OPERATING CORPORATION']
filtered_list = combined_data[combined_data.OperatorName.isin(filtered_company_list)]
filtered_list.head(100)